In [ ]:
import numpy as np
from gould_2026.datasets import Zong22Dataset, Odoherty21Dataset
from gould_2026.sim_stim import SimStimConfig, run_simulations, StimResponseModelType
import matplotlib.pyplot as plt
from gould_2026.stim_designer import StimDesigner, OptimizationMethod
from gould_2026.stim_designer import _linear_u_to_s, Partial
from gould_2026.plotting import Palette, LINEWIDTH, paper_plot_context
from itertools import cycle

In [ ]:
output = None
condition = 'positive_constrained'

In [ ]:
if condition == 'constrained':
    opt_method, scale_factor = OptimizationMethod.LBFGS, .1
elif condition == 'positive_constrained':
    opt_method, scale_factor = OptimizationMethod.LBFGS_POSITIVE_CONSTRAINED, .1
elif condition == 'sparse_constrained':
    opt_method, scale_factor = OptimizationMethod.LBFGS_SPARSE_CONSTRAINED, .1
elif condition == 'unconstrained':
    opt_method, scale_factor = OptimizationMethod.LBFGS_UNCONSTRAINED, .05
else:
    raise ValueError()


In [ ]:
rng = np.random.default_rng(0)
d = Odoherty21Dataset()
data = d.neural_data

to_run = {
    'learning from stim': SimStimConfig(
        stim_magnitude=0,
        optimization_method=opt_method,
        u_to_s_model_type=StimResponseModelType.IDENTITY,
        exit_time=np.inf,
        smoothing_tau=1,
        centerer_init_size=8 * 25,
        initial_nostim_period=30,
        stim_timing_method='isi',
        attempt_correction=True,
        heed_stimuli=True,
        show_tqdm=True,
        prosvd_k=10
    ),
}

sims = run_simulations(data, rng, to_run=to_run, n_runs=1, show_tqdm=False)

sim = sims['learning from stim'][0]


In [ ]:
%matplotlib inline

with paper_plot_context():
    fig, axs = plt.subplots(figsize=(1.7,1.7), layout='constrained', squeeze=False)


    stim_designer = StimDesigner(should_log=True, rng_seed=1)

    theta=.1
    i=5
    l=1
    r=0

    latents = sim.log['latents'].slice_by_time(slice(30,None))
    ax = axs[0,0]
    ax.plot(latents[:, 0], latents[:, 1], alpha=.1, color='k')

    center_t = sim.log['stim_intended_samples'].t[i]
    latents = sim.log['latents'].slice_by_time(slice(center_t-l,center_t+r))
    line = ax.plot(latents[:-1, 0], latents[:-1, 1], color='k')
    stim_s = sim.log['stim_intended_samples'].slice_by_time(slice(center_t-l,center_t+r)).t - latents.dt
    latents_s = latents.slice_by_time(stim_s).reshape((-1, latents.shape[1]))
    ax.plot(latents_s[:, 0], latents_s[:, 1], '.', color='g')



    u = sim.stim_designer.log[i]['u']
    v = sim.stim_designer.log[i]['v']

    s_s = []
    for theta in np.linspace(0, 2*np.pi, 21)[:-1] - 0.11:
        v = 0 * v
        v[0,0] = np.cos(theta)
        v[1,0] = np.sin(theta)

        equivalent_projection_matrix = sim.stim_designer.log[i]['equiv_proj_mat']
        previous_us = sim.stim_designer.log[i]['previous_us']
        previous_us = np.array(previous_us) if (previous_us is not None) else None

        u_to_s_function=Partial(_linear_u_to_s, A=equivalent_projection_matrix.T, stim_magnitude=1)

        stim_designer.optimization_method = opt_method
        new_u = stim_designer.design_stim(v=v, u_dimension=u.size, u_to_s_function=u_to_s_function, equivalent_projection_matrix=equivalent_projection_matrix, previous_us=previous_us)

        s = u_to_s_function(new_u) * scale_factor
        s_s.append(s)

    for s in s_s:
        axs[0,0].annotate(
            '',
            xytext=(latents_s[0,0], latents_s[0,1]),
            xy=(latents_s[0,0]+s[0], latents_s[0,1]+s[1]),
            arrowprops=dict(color=Palette.s_designed, width=LINEWIDTH, clip_on=True, headwidth=7*LINEWIDTH, headlength=7*LINEWIDTH),
            annotation_clip=False
        )


    axs[0,0].axis('equal')
    axs[0,0].axis('off')
    axs[0,0].set_xlim(np.array([-1,1])*.3 + latents_s[:, 0])
    axs[0,0].set_ylim(np.array([-1,1])*.3 + latents_s[:, 1])



    if output is not None:
        fig.savefig(output)

In [ ]:
basis = equivalent_projection_matrix[:,:2]

In [ ]:
# warning: this is AI generated, need to check this

def zonotope_polygon_2d(generators, lo=-1.0, hi=1.0):
    generators = np.asarray(generators, dtype=float)
    w = (hi - lo) * generators

    # Flip generators pointing into the lower half-plane so every vector has
    # an angle in [0, pi) -- this puts the zonotope into "canonical" form.
    flip = (w[:, 1] < 0) | ((w[:, 1] == 0) & (w[:, 0] < 0))
    h = np.where(flip[:, None], -w, w)

    order = np.argsort(np.arctan2(h[:, 1], h[:, 0]))
    h_sorted = h[order]

    cum = np.cumsum(h_sorted, axis=0)
    top = cum[-1]
    upper = np.vstack([np.zeros(2), cum])       # 0 -> top, n+1 vertices
    lower_middle = top - cum[:-1]               # top -> 0, n-1 more vertices

    verts = np.vstack([upper, lower_middle])
    base_shift = lo * generators.sum(axis=0) + w[flip].sum(axis=0)
    return verts + base_shift

In [ ]:
lo, hi = (0., 1.) if condition == 'positive_constrained' else (-1., 1.)
zonotope_verts = zonotope_polygon_2d(basis, lo=lo, hi=hi) * scale_factor + latents_s[0][:2]


axs[0, 0].add_patch(plt.Polygon(zonotope_verts, closed=True, facecolor=Palette.s_designed, edgecolor='none', alpha=.25, zorder=0))
fig

# plt.plot(zonotope_verts[:,0], zonotope_verts[:,1], '.')


In [ ]:
z = zonotope_polygon_2d(basis, lo=0, hi=1)
plt.plot(z[:,0], z[:,1])

z = zonotope_polygon_2d(basis, lo=-1, hi=1)
plt.plot(z[:,0], z[:,1])
